# 📓 Notebook 03 : Entraînement Modulaire Stage 2 (CADx) & Reprise Automatique (Restart-Safe)
**Projet :** LAAFI_AI IVA Engine (Version 2.0)
**Règles appliquées :** `ml-best-practices` & `notebook-guidance`.
**Objectifs :**
- Configurer l'environnement GPU/Drive de manière résiliente (force_remount) et la reproductibilité (Seed = 42).
- **Système Restart-Safe** : Détection et reprise automatique depuis `latest_checkpoint.pt` en cas de déconnexion Colab.
- Entraîner le classificateur ConvNeXt-Base en précision mixte (AMP + AdamW + Cosine Scheduler).
- Optimiser dynamiquement la Focal Loss et le seuil de décision clinique (Sensibilité >= 95%).
- Générer et mettre à jour automatiquement les rapports CSV, JSON et la courbe d'apprentissage après chaque epoch.

## 1. Initialisation de l'Environnement Colab & Reproductibilité (Montage Drive Résilient)
Cette cellule remonte automatiquement le Google Drive (`force_remount=True`), ajuste le `sys.path` pour importer le package local `src` et fixe la graine globale à 42.

In [ ]:
# 1. Setup Répertoire, sys.path & Reproductibilité (Drive Mount Résilient)
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    project_path = "/content/drive/MyDrive/LAAFI_AI_IVA"
    os.makedirs(project_path, exist_ok=True)
    os.chdir(project_path)
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    print(f"✅ Répertoire de travail Colab activé : {os.getcwd()}")
else:
    print(f"💻 Environnement Local détecté : {os.getcwd()}")

PROJECT_ROOT = Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import seed_everything
seed_everything(42)
print("🔒 Seed globale fixée à 42.")

## 2. Diagnostics Matériels (GPU & VRAM) & Statut des Checkpoints
Vérifie la présence du GPU Tesla T4 et contrôle l'existence d'un checkpoint de reprise (`latest_checkpoint.pt`) sur le Drive.

In [ ]:
# 2. Hardware & Checkpoint Status
import torch
print(f"PyTorch Version : {torch.__version__}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🚀 GPU Détecté : {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ Aucun GPU détecté !")

checkpoint_path = "./models/checkpoints/latest_checkpoint.pt"
if os.path.exists(checkpoint_path):
    print(f"🔄 Checkpoint de reprise détecté sur le Drive : {checkpoint_path}")
else:
    print("🆕 Aucun checkpoint de reprise. Un nouvel entraînement démarrera à l'epoch 1.")

## 3. Lancement / Reprise Automatique (Restart-Safe) de l'Entraînement
Exécute le moteur d'entraînement principal `train_laafi_ai_model`. Si une déconnexion survient sur Colab, la relance de cette cellule reprendra automatiquement là où elle s'était arrêtée.

In [ ]:
# 3. Lancement du Moteur d'Entraînement Stage 2 avec Reprise Automatique
from src.train import train_laafi_ai_model

config_file = "./config/config.yaml"
print(f"🚀 Exécution du moteur d'entraînement (Config: {config_file})")
train_laafi_ai_model(config_path=config_file)

## 4. Visualisation Dynamique du Rapport et des Courbes d'Apprentissage
Affiche le tableau des métriques sauvegardé à chaque epoch (`training_history.csv`) et le graphique de suivi (`learning_curves.png`).

In [ ]:
# 4. Affichage du Rapport et des Figures
import pandas as pd
from IPython.display import Image, display

history_csv = "./outputs/reports/training_history.csv"
curve_img = "./outputs/figures/learning_curves.png"

if os.path.exists(history_csv):
    df_history = pd.read_csv(history_csv)
    print("📊 Tableau des métriques obtenues par Epoch :")
    display(df_history)
else:
    print("⚠️ Aucun fichier d'historique trouvé pour l'instant.")

if os.path.exists(curve_img):
    print("🎨 Courbes de Loss, AUC et Score F2 :")
    display(Image(filename=curve_img))
else:
    print("⚠️ Courbe d'apprentissage non disponible.")

### Data Analysis Key Findings
- **Continuité Garantie (Restart-Safe)** : Grâce au montage résilient du Drive (`force_remount=True`) et au chargement de `latest_checkpoint.pt`, l'entraînement peut être interrompu et repris à tout moment.
- **Ajustement dynamique du Seuil** : Le seuil de décision est calibré par epoch afin d'atteindre une Sensibilité (Recall) >= 95% conformément aux exigences de sécurité clinique SaMD.

### Insights or Next Steps
- Une fois l'entraînement complété (30 epochs), ouvrir le **Notebook 04 (`04_eval_inference.ipynb`)** pour l'évaluation test finale et la matrice de confusion.